# WaveGuard Colab — yolo26s 파인튜닝 (Python API만 사용)

이 노트북은 `!yolo` / `python -m ultralytics` 를 **쓰지 않습니다**.
Colab에서 CLI가 깨지는 문제를 피하기 위해 `from ultralytics import YOLO` 만 사용합니다.

## 준비
1. 로컬에서 `finetune/make_dataset.py` 로 만든 `gwangalli_dataset.zip` 또는 `gwangalli_colab.zip` 을 Drive에 업로드
2. Colab **런타임 → 런타임 유형 변경 → GPU (T4)**
3. 아래 셀을 **위에서부터** 실행

학습 후 `best.pt` → 로컬 `vision/models/yolo26s_beach_ft.pt` 에 넣고 서버 재시작.

In [ ]:
# 1) GPU 확인
!nvidia-smi

In [ ]:
# 2) ultralytics 설치 + API 확인 (CLI 사용 안 함)
!pip -q install -U ultralytics
import torch
from ultralytics import YOLO
import ultralytics
ultralytics.checks()
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — 런타임을 GPU로 바꾸세요')
print('YOLO API OK')

In [ ]:
# 3) Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4) 데이터셋 준비 — 아래 ZIP 경로 중 본인 환경에 맞게 하나만 쓰세요
import os, zipfile, shutil

# A) make_dataset.py 로 만든 zip (권장)
ZIP = '/content/drive/MyDrive/gwangalli/gwangalli_dataset.zip'
# B) pack_for_colab.ps1 패키지 zip 을 쓰는 경우 아래를 쓰고 A를 주석
# ZIP = '/content/drive/MyDrive/gwangalli_colab.zip'

assert os.path.exists(ZIP), f'zip 없음: {ZIP}'

if ZIP.endswith('gwangalli_colab.zip'):
    # 전체 vision 패키지
    if os.path.exists('/content/pkg'):
        shutil.rmtree('/content/pkg')
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content/pkg')
    DATA = '/content/pkg/vision/finetune/dataset/data.yaml'
    ROOT = '/content/pkg/vision/finetune/dataset'
    # Colab 절대경로로 data.yaml 고정
    open(DATA, 'w').write(
        f'path: {ROOT}\ntrain: train/images\nval: val/images\n'
        'names:\n  0: person\n  1: tube\n'
    )
else:
    DST = '/content/gwangalli'
    if os.path.exists(DST):
        shutil.rmtree(DST)
    os.makedirs(DST, exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DST)
    DATA = os.path.join(DST, 'data.yaml')
    txt = open(DATA, encoding='utf-8').read().replace('path: .', f'path: {DST}')
    open(DATA, 'w', encoding='utf-8').write(txt)

print('data.yaml:\n', open(DATA, encoding='utf-8').read())
root = os.path.dirname(DATA)
print('train:', len(os.listdir(os.path.join(root, 'train/images'))))
print('val:', len(os.listdir(os.path.join(root, 'val/images'))))

In [ ]:
# 5) 학습 — Python API only (여기 !yolo 쓰면 안 됨)
from ultralytics import YOLO
import os

DATA = DATA if 'DATA' in dir() else '/content/gwangalli/data.yaml'
assert os.path.exists(DATA), DATA

BASE = 'yolo26s.pt'
try:
    model = YOLO(BASE)
except Exception as e:
    print('yolo26s 실패 → yolov8s:', e)
    BASE = 'yolov8s.pt'
    model = YOLO(BASE)

print('base:', BASE)
results = model.train(
    data=DATA,
    epochs=100,
    imgsz=1024,   # OOM이면 768
    batch=8,      # OOM이면 4
    device=0,
    patience=30,
    close_mosaic=10,
    hsv_v=0.5,
    degrees=0.0,
    translate=0.05,
    scale=0.3,
    fliplr=0.5,
    project='/content/runs',
    name='gwangalli',
    exist_ok=True,
)
print('DONE best=', '/content/runs/gwangalli/weights/best.pt')

In [ ]:
# 6) 검증
from ultralytics import YOLO
BEST = '/content/runs/gwangalli/weights/best.pt'
model = YOLO(BEST)
metrics = model.val(data=DATA, imgsz=1024, device=0)
print('mAP50:', round(float(metrics.box.map50), 4))
print('mAP50-95:', round(float(metrics.box.map), 4))
print('P:', round(float(metrics.box.mp), 4), 'R:', round(float(metrics.box.mr), 4))

In [ ]:
# 7) Drive 저장 + 브라우저 다운로드
import shutil, os
from google.colab import files

BEST = '/content/runs/gwangalli/weights/best.pt'
os.makedirs('/content/drive/MyDrive/gwangalli', exist_ok=True)
out = '/content/drive/MyDrive/gwangalli/yolo26s_beach_ft.pt'
shutil.copy(BEST, out)
print('Drive:', out, os.path.getsize(out)//1024, 'KB')
files.download(BEST)
print('로컬에 저장: vision/models/yolo26s_beach_ft.pt 후 서버 재시작')